# AI & ML Engineering Concepts for Data Engineers
## ML Pipelines, Feature Stores, Model Serving, LLMs & RAG

**What you'll learn:**
- The ML lifecycle and where DE fits in
- Feature engineering and feature stores (Databricks Feature Store, Feast)
- ML pipeline architecture (training, validation, deployment)
- Model serving patterns (batch, real-time, streaming)
- MLOps: experiment tracking, model registry, A/B testing
- Embeddings and vector databases (Pinecone, Weaviate, Chroma)
- RAG (Retrieval-Augmented Generation) architecture
- LLM integration for data pipelines (classification, extraction, enrichment)
- Data quality for ML (training-serving skew, data drift)
- Databricks ML ecosystem (MLflow, Feature Store, Model Serving)
- Interview questions

**Estimated Time:** 7-9 hours

---

> As a DE in 2026, you're expected to build the DATA INFRASTRUCTURE that
> powers ML models. You don't need to train models, but you need to understand
> how data flows into and out of ML systems.

---

---
# Section 1: The ML Lifecycle -- Where Data Engineering Fits

## The ML Lifecycle (and Your Role)

```
┌─────────────────────────────────────────────────────────────────────┐
│                        ML LIFECYCLE                                   │
│                                                                       │
│  ┌─────────┐    ┌──────────┐    ┌─────────┐    ┌──────────┐        │
│  │  DATA    │───>│ FEATURE  │───>│ MODEL   │───>│ MODEL    │        │
│  │COLLECTION│    │ENGINEERING│    │TRAINING │    │ SERVING  │        │
│  │          │    │          │    │          │    │          │        │
│  │ Ingest   │    │ Transform│    │ Train    │    │ Predict  │        │
│  │ Store    │    │ Store in │    │ Validate │    │ Monitor  │        │
│  │ Quality  │    │ Feature  │    │ Register │    │ Retrain  │        │
│  │ Catalog  │    │ Store    │    │          │    │          │        │
│  └─────────┘    └──────────┘    └─────────┘    └──────────┘        │
│  ^^^^^^^^^^^    ^^^^^^^^^^^^                    ^^^^^^^^^^^^         │
│  YOUR JOB       YOUR JOB                        YOUR JOB            │
│  (70% DE)       (shared w/ DS)                   (infra)            │
└─────────────────────────────────────────────────────────────────────┘
```

## What DEs Build for ML Teams

| What You Build | Why ML Teams Need It | Tools |
|---------------|---------------------|-------|
| **Data pipelines** | Clean, reliable training data | Spark, Airflow, Delta Lake |
| **Feature pipelines** | Compute features at scale | Spark, Flink, dbt |
| **Feature store** | Serve features for training AND inference | Databricks FS, Feast, Tecton |
| **Data quality** | Prevent garbage in → garbage out | Great Expectations, Deequ |
| **Model input tables** | Pre-joined, pre-aggregated training data | Delta Lake, SQL |
| **Serving infrastructure** | Low-latency feature lookup | Redis, DynamoDB |
| **Monitoring pipelines** | Detect data/model drift | Custom + ML tools |
| **Labeling pipelines** | Manage ground truth data | Custom + Label Studio |

## The 80/20 Rule of ML Projects

```
80% of ML project time = DATA WORK (that's YOU!)
  - Finding and collecting data
  - Cleaning and transforming
  - Feature engineering
  - Building reliable pipelines
  - Monitoring data quality

20% of ML project time = MODEL WORK (that's the ML engineer)
  - Choosing architecture
  - Training and tuning
  - Evaluation
```

---
# Section 2: Feature Engineering & Feature Stores

## What is a Feature Store?

A feature store is a centralized repository for ML features that:
- Computes features consistently (same logic for training AND serving)
- Stores features with versioning and lineage
- Serves features at low latency for real-time predictions
- Prevents training-serving skew (the #1 ML production bug)

```
                    FEATURE STORE
┌──────────────────────────────────────────────────┐
│                                                    │
│  OFFLINE STORE          │  ONLINE STORE            │
│  (training data)        │  (serving data)          │
│                         │                          │
│  Delta Lake / S3        │  Redis / DynamoDB        │
│  Full history           │  Latest values only      │
│  Batch reads            │  <10ms lookups           │
│                         │                          │
│  ┌─────────────┐       │  ┌─────────────┐        │
│  │ customer_id │       │  │ customer_id │        │
│  │ avg_order_30d│      │  │ avg_order_30d│       │
│  │ total_orders │       │  │ total_orders │        │
│  │ days_since   │       │  │ days_since   │        │
│  │ last_purchase│       │  │ last_purchase│        │
│  └─────────────┘       │  └─────────────┘        │
│                         │                          │
│  Used for: TRAINING     │  Used for: INFERENCE     │
│  (batch, large volumes) │  (real-time, per request)│
└──────────────────────────────────────────────────┘
         ▲                           ▲
         │                           │
    Feature Pipeline             Feature Pipeline
    (Spark, daily batch)         (Flink, streaming)
```

## Feature Types

| Type | Description | Example | Computation |
|------|-------------|---------|-------------|
| **Raw** | Direct from source | customer_age, city | None (pass-through) |
| **Aggregated** | Windows/rollups | avg_order_last_30d | Spark batch or streaming |
| **Derived** | Calculated | days_since_last_order | Simple transform |
| **Crossed** | Interaction of features | city_x_product_category | Join + compute |
| **Embedding** | Vector representation | user_embedding_128d | ML model output |
| **Real-time** | Just-in-time | current_cart_value | Streaming or request-time |

In [ ]:
# Feature engineering patterns in PySpark
print("FEATURE ENGINEERING FOR ML")
print("=" * 60)

print("""
# Typical feature pipeline (what DE builds):

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Raw data
orders = spark.table("techmart_dw.orders")
customers = spark.table("techmart_dw.customers")

# WINDOW-BASED FEATURES (most common pattern)
w30 = Window.partitionBy("customer_id").orderBy("order_date") \
    .rangeBetween(-30*86400, 0)  # 30 days
w90 = Window.partitionBy("customer_id").orderBy("order_date") \
    .rangeBetween(-90*86400, 0)  # 90 days

customer_features = (orders
    .filter(col("status") == "completed")
    .groupBy("customer_id")
    .agg(
        # Recency
        datediff(current_date(), max("order_date")).alias("days_since_last_order"),
        # Frequency
        count("*").alias("total_orders"),
        countDistinct(date_trunc("month", col("order_date"))).alias("active_months"),
        # Monetary
        sum("total_amount").alias("lifetime_revenue"),
        avg("total_amount").alias("avg_order_value"),
        stddev("total_amount").alias("order_value_stddev"),
        # Behavioral
        countDistinct("payment_method").alias("payment_methods_used"),
        countDistinct("order_source").alias("channels_used"),
        # Trend
        sum(when(col("order_date") >= date_sub(current_date(), 30),
                 col("total_amount"))).alias("revenue_last_30d"),
        sum(when(col("order_date") >= date_sub(current_date(), 90),
                 col("total_amount"))).alias("revenue_last_90d"),
    )
    .withColumn("revenue_trend",
        col("revenue_last_30d") / (col("revenue_last_90d") / 3))  # Recent vs average
)

# WRITE TO FEATURE STORE (Databricks):
# customer_features.write.format("delta").mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable("feature_store.customer_features")

# In Databricks Feature Store:
# from databricks.feature_store import FeatureStoreClient
# fs = FeatureStoreClient()
# fs.create_table("customer_features", primary_keys=["customer_id"],
#                 df=customer_features, description="Customer ML features")
""")

print("KEY PRINCIPLES:")
print("  1. Same code computes features for training AND serving")
print("  2. Features are versioned (point-in-time correctness)")
print("  3. Feature freshness matches use case (daily for batch, real-time for serving)")
print("  4. Document every feature (name, description, owner, SLA)")

---
# Section 3: ML Pipeline Architecture

## Training Pipeline (Batch -- what DE builds)

```
┌─────────┐    ┌──────────┐    ┌─────────┐    ┌──────────┐    ┌────────┐
│ Feature │───>│ Training │───>│Validation│───>│  Model   │───>│Deploy/ │
│  Store  │    │  Dataset │    │          │    │ Registry │    │ Serve  │
│(offline)│    │(join +   │    │ Metrics  │    │(MLflow)  │    │        │
│         │    │ label)   │    │ Compare  │    │ Version  │    │ A/B    │
│         │    │ Split    │    │ Approve  │    │ Stage    │    │ Test   │
└─────────┘    └──────────┘    └─────────┘    └──────────┘    └────────┘
    DE job       DE job          ML Eng         MLOps           Platform
```

## Serving Pipeline (Real-Time)

```
REQUEST: "Should we approve this transaction?"
    │
    ▼
┌───────────────────────────────────────────────────────────────┐
│ MODEL SERVING ENDPOINT                                         │
│                                                                 │
│  1. Receive request (transaction_id, customer_id, amount)      │
│  2. Fetch features from ONLINE STORE:                          │
│     - customer_avg_transaction (Redis/DynamoDB): 5ms            │
│     - customer_location_history: 3ms                            │
│     - merchant_fraud_rate: 2ms                                  │
│  3. Combine request features + stored features                  │
│  4. Run model inference: 10ms                                   │
│  5. Return prediction: {fraud_probability: 0.87, decision: "block"} │
│                                                                 │
│  Total latency: <50ms (SLA: 100ms p99)                         │
└───────────────────────────────────────────────────────────────┘
    │
    ▼
DE builds: feature pipelines that keep online store fresh
```

## MLflow (Experiment Tracking + Model Registry)

```python
import mlflow

# Experiment tracking (ML Engineer does this, DE understands it)
with mlflow.start_run():
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("features_version", "v2.3")
    mlflow.log_metric("auc", 0.94)
    mlflow.log_metric("precision", 0.91)
    mlflow.sklearn.log_model(model, "fraud_detector")

# Model Registry (MLOps manages, DE integrates with)
# Stages: None → Staging → Production → Archived
# DE ensures: training data pipeline feeds the right version
```

---
# Section 4: Embeddings, Vector Databases & RAG

## What are Embeddings?

Embeddings convert data (text, images, users) into dense numerical vectors
that capture semantic meaning. Similar items have vectors close together.

```
"Data engineer"  → [0.82, 0.15, 0.93, 0.41, ...] (768 dimensions)
"Data scientist" → [0.79, 0.18, 0.88, 0.39, ...] (nearby!)
"Chef"           → [0.12, 0.85, 0.21, 0.67, ...] (far away)

Similarity = cosine_similarity(vec_A, vec_B)
  DE vs DS: 0.95 (very similar)
  DE vs Chef: 0.12 (very different)
```

## Vector Databases

| Database | Type | Best For |
|----------|------|----------|
| **Pinecone** | Managed cloud | Production, simple API |
| **Weaviate** | Open-source | Self-hosted, rich queries |
| **Chroma** | Open-source | Lightweight, prototyping |
| **Milvus** | Open-source | Large scale (billions of vectors) |
| **pgvector** | Postgres extension | Already using Postgres |
| **Databricks Vector Search** | Managed (Databricks) | Lakehouse integration |

## RAG (Retrieval-Augmented Generation)

```
USER QUERY: "What is our data retention policy?"
    │
    ▼
1. EMBED the query → vector [0.5, 0.3, ...]
    │
    ▼
2. SEARCH vector database for similar document chunks
    │  Returns: top 5 most relevant chunks from company docs
    ▼
3. CONSTRUCT PROMPT:
    "Given this context: [relevant chunks]
     Answer this question: What is our data retention policy?"
    │
    ▼
4. LLM generates answer grounded in YOUR data
    │
    ▼
RESPONSE: "According to our data governance policy (doc v2.1),
           PII data is retained for 90 days, aggregated metrics
           for 7 years, and raw logs for 30 days."
```

## DE Role in RAG Systems

| Component | What DE Builds |
|-----------|---------------|
| Document pipeline | Ingest docs, chunk, clean, embed, index |
| Embedding pipeline | Run embedding model on new/updated documents |
| Vector index maintenance | Keep index fresh, handle updates/deletes |
| Monitoring | Track retrieval quality, latency, freshness |
| Data connectors | Integrate with Confluence, SharePoint, S3, databases |

In [ ]:
# AI/ML concepts for DE
print("AI ENGINEERING CONCEPTS FOR DE")
print("=" * 60)

# Embedding simulation
import math

def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

# Simulated embeddings (in reality these are 768+ dimensions)
embeddings = {
    "data engineer": [0.82, 0.15, 0.93, 0.41, 0.67, 0.23, 0.88, 0.55],
    "data scientist": [0.79, 0.18, 0.88, 0.39, 0.71, 0.28, 0.84, 0.52],
    "ML engineer": [0.75, 0.22, 0.85, 0.45, 0.73, 0.35, 0.80, 0.48],
    "frontend developer": [0.30, 0.80, 0.25, 0.65, 0.20, 0.75, 0.35, 0.60],
    "chef": [0.10, 0.05, 0.15, 0.90, 0.08, 0.88, 0.12, 0.95],
}

print("\n1. EMBEDDING SIMILARITY (cosine):")
query = "data engineer"
print(f"   Query: '{query}'")
print(f"   {'Term':<25} {'Similarity':<12} {'Match?'}")
print(f"   {'-'*50}")
for term, vec in embeddings.items():
    if term != query:
        sim = cosine_similarity(embeddings[query], vec)
        match = "HIGH" if sim > 0.9 else "MEDIUM" if sim > 0.5 else "LOW"
        print(f"   {term:<25} {sim:.4f}       {match}")

# RAG pipeline components
print("\n2. RAG PIPELINE (what DE builds):")
print("""
   Document Pipeline:
     Raw docs → Chunk (512 tokens) → Clean → Embed → Store in Vector DB

   Query Pipeline:
     User query → Embed → Vector search (top-k) → Format context → LLM → Answer

   DE Responsibilities:
     - Document ingestion pipeline (new docs, updates, deletes)
     - Embedding computation (batch for bulk, streaming for new)
     - Vector index management (rebuild, optimize, shard)
     - Monitoring (retrieval quality, stale docs, latency)
""")

# Training-serving skew
print("3. TRAINING-SERVING SKEW (the #1 ML production bug):")
print("""
   Problem: Feature computed DIFFERENTLY for training vs serving

   Training: avg_order_value = SUM(amount) / COUNT(*)
             (computed on historical data with Spark)

   Serving:  avg_order_value = SUM(last_10) / 10
             (computed at request time with different logic!)

   Result: Model trained on one distribution, serves on another = bad predictions!

   Fix: Feature Store ensures SAME computation for both.
   DE responsibility: Build feature pipelines that are consistent.
""")

---
# Section 5: LLM Integration for Data Pipelines

## How DEs Use LLMs in Pipelines

| Use Case | Input | LLM Task | Output |
|----------|-------|----------|--------|
| **Classification** | Customer feedback text | Categorize sentiment | positive/negative/neutral |
| **Extraction** | Invoice PDFs | Extract structured fields | JSON (amount, date, vendor) |
| **Enrichment** | Product descriptions | Generate categories/tags | Structured metadata |
| **Quality** | Address strings | Standardize/validate | Cleaned addresses |
| **Summarization** | Long documents | Create summaries | Short text |
| **Code generation** | Schema description | Generate SQL/dbt models | Working code |

## Architecture: LLM in a Data Pipeline

```
Bronze (raw)          Silver (cleaned)         Gold (enriched)
┌──────────┐         ┌──────────────┐         ┌──────────────┐
│ Raw text │ ──ETL──>│ Cleaned text │──LLM──> │ Structured   │
│ feedback │         │ deduplicated │  batch   │ sentiment,   │
│ messy    │         │ normalized   │  enrich  │ categories,  │
│          │         │              │          │ entities     │
└──────────┘         └──────────────┘         └──────────────┘

Key considerations:
  - Batch LLM calls (not per-record -- too expensive!)
  - Cache results (same input → same output)
  - Handle rate limits (exponential backoff)
  - Cost management ($0.01-$0.03 per 1K tokens)
  - Fallback for failures (retry, default value)
  - Quality monitoring (spot-check LLM outputs)
```

## Databricks AI Ecosystem

| Component | Purpose |
|-----------|---------|
| **MLflow** | Experiment tracking, model registry, deployment |
| **Feature Store** | Centralized feature management |
| **Model Serving** | Real-time inference endpoints |
| **Vector Search** | Managed vector database for RAG |
| **Foundation Model APIs** | Access to LLMs (DBRX, Llama, etc.) |
| **AI Playground** | Test LLMs interactively |
| **Genie** | Natural language to SQL/data |
| **AI Functions** | SQL functions powered by LLMs |

In [ ]:
# MLOps and model monitoring concepts
print("MLOps CONCEPTS FOR DATA ENGINEERS")
print("=" * 60)

print("""
MODEL MONITORING (what DE pipelines track):

1. DATA DRIFT (input distribution changed):
   - Feature distributions shift over time
   - Example: avg_order_value was $50 in training, now $200 (inflation/bug?)
   - DE builds: comparison pipeline (current vs training baseline)
   - Tool: Evidently AI, custom Spark jobs, Databricks Lakehouse Monitoring

2. PREDICTION DRIFT (output distribution changed):
   - Model predicting differently than before
   - Example: fraud model suddenly flags 50% of transactions (was 2%)
   - DE builds: prediction logging + monitoring pipeline

3. FEATURE FRESHNESS:
   - Features in online store must be up-to-date
   - Example: "last_login" feature is 24h stale = wrong predictions
   - DE builds: freshness monitoring, SLA tracking

4. TRAINING-SERVING SKEW:
   - Same feature computed differently in training vs serving
   - DE builds: consistency tests between offline and online stores

MONITORING PIPELINE ARCHITECTURE:
  Predictions logged → Delta Lake → Drift detection job (daily)
                                  → Alert if drift > threshold
                                  → Trigger retraining pipeline
""")

# Cost estimation for LLM in pipelines
print("\nLLM COST ESTIMATION:")
print(f"  {'Records':<12} {'Avg Tokens':<12} {'Model':<15} {'Cost'}")
print(f"  {'-'*55}")
scenarios = [
    (10000, 200, "GPT-4o-mini", 0.00015),
    (10000, 200, "GPT-4o", 0.0025),
    (100000, 500, "GPT-4o-mini", 0.00015),
    (100000, 500, "GPT-4o", 0.0025),
    (1000000, 200, "GPT-4o-mini", 0.00015),
]
for records, tokens, model, cost_per_1k in scenarios:
    total_tokens = records * tokens
    cost = total_tokens / 1000 * cost_per_1k
    print(f"  {records:<12,} {tokens:<12} {model:<15} ${cost:,.2f}")

print("\n  Rule: Use cheapest model that meets quality bar.")
print("  Batch processing + caching = major cost savings.")

In [ ]:
print("\nAI ENGINEERING INTERVIEW QUESTIONS FOR DE")
print("=" * 60)

questions = [
    {
        "q": "What is a feature store and why do DEs care?",
        "a": "A feature store is a centralized repo for ML features with offline (training) and online (serving) stores. DEs care because: (1) we build the pipelines that compute features, (2) we ensure consistency between training and serving, (3) we manage freshness SLAs, (4) we prevent training-serving skew which is the #1 ML production bug."
    },
    {
        "q": "How would you build a RAG pipeline for company documents?",
        "a": "1) Ingest pipeline: crawl docs from Confluence/SharePoint/S3, chunk into 512-token segments, 2) Embedding pipeline: run embedding model (batch), store vectors in vector DB, 3) Query pipeline: embed user query, vector similarity search for top-k chunks, 4) Generation: pass chunks as context to LLM, 5) Monitoring: track retrieval quality, freshness, cost."
    },
    {
        "q": "How do you handle data drift in production ML systems?",
        "a": "Build a monitoring pipeline: (1) Log all prediction inputs to Delta Lake, (2) Daily job compares current feature distributions vs training baseline (KS test, PSI), (3) Alert if drift exceeds threshold, (4) Optionally trigger automated retraining. The DE owns the logging and comparison pipelines; ML engineer owns the retraining logic."
    },
    {
        "q": "What is training-serving skew and how do you prevent it?",
        "a": "When features are computed differently for training (batch, historical) vs serving (real-time, current). Prevention: use a feature store that guarantees same computation logic for both. The offline store provides training data, the online store serves the same features at inference time. Same code, different materialization."
    },
    {
        "q": "How would you integrate LLMs into a data pipeline?",
        "a": "Batch enrichment pattern: (1) Silver table with cleaned text, (2) Spark job calls LLM API in batches (with rate limiting, retries, caching), (3) Store structured outputs in Gold table. Key considerations: cost management (cheapest sufficient model), caching (same input = same output), fallbacks, quality monitoring (sample and review outputs)."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: AI Engineering for DE Cheat Sheet

## What DEs Need to Know vs Don't

| Must Know | Nice to Know | Don't Need |
|-----------|-------------|-----------|
| Feature pipelines | Model architectures | Neural network math |
| Feature stores (read/write) | Hyperparameter tuning | Loss function derivation |
| MLflow basics (registry, tracking) | Evaluation metrics (AUC, F1) | Backpropagation |
| Embedding pipelines | Vector similarity search | Training optimizers |
| RAG architecture | Prompt engineering | Fine-tuning LLMs |
| Data/model drift monitoring | A/B testing for models | Building custom models |
| Training data preparation | Model serving infra | GPU programming |
| Batch LLM integration | Streaming inference | RLHF |

## Key Patterns

| Pattern | DE Responsibility | Tool |
|---------|------------------|------|
| Feature pipeline | Compute features from raw data | Spark, Flink, dbt |
| Feature store | Populate offline + online stores | Databricks FS, Feast |
| Training dataset | Point-in-time join features + labels | Spark SQL, Delta |
| Model monitoring | Log predictions, detect drift | Custom pipelines |
| RAG ingestion | Chunk, embed, index documents | Spark + Vector DB |
| LLM enrichment | Batch API calls with quality checks | Python + API |

---
*AI & ML Engineering Concepts for Data Engineers*